# Comparação: Logistic Regression vs Softmax Regression

Comparação empírica dos dois algoritmos no dataset multiclass **multiclass_classification**

In [77]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Importar os algoritmos
from src.logistic_regression import LogisticRegression
from src.softmax_logistic_regression import SoftmaxRegression

sns.set_theme(style="whitegrid")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Carregamento e Classificação dos Datasets Multiclass

In [78]:
# Carregar TODOS os datasets do multiclass_classification
multiclass_path = "datasets/multiclass_classification"

datasets = []
datasets_int_classes = []      # Apenas classes inteiras
datasets_non_int_classes = []  # Classes não-inteiras

if os.path.exists(multiclass_path):
    csv_files = [f for f in os.listdir(multiclass_path) if f.endswith('.csv')]
    print(f"Total de arquivos encontrados: {len(csv_files)}\n")
    
    for csv_file in sorted(csv_files):
        try:
            filepath = os.path.join(multiclass_path, csv_file)
            df_data = pd.read_csv(filepath)
            
            X = df_data.iloc[:, :-1].values.astype(str)
            y = df_data.iloc[:, -1]  # Manter tipos originais
            
            # Converter para números sequenciais para compatibilidade
            y_numeric = pd.factorize(y)[0]
            
            # Verificar tipo de classes original
            classes_unique = y.unique()
            all_int = all(isinstance(c, (int, np.integer)) for c in classes_unique)
            
            dataset_info = {
                'name': csv_file,
                'X': X,
                'y': y_numeric,
                'y_original': y,
                'n_samples': X.shape[0],
                'n_features': X.shape[1],
                'n_classes': len(np.unique(y_numeric)),
                'classes_are_int': all_int,
                'classes_original': list(classes_unique)
            }
            
            datasets.append(dataset_info)
            
            if all_int:
                datasets_int_classes.append(dataset_info)
            else:
                datasets_non_int_classes.append(dataset_info)
                
        except Exception as e:
            print(f"Erro ao carregar {csv_file}: {e}")
else:
    print(f"Caminho não encontrado: {multiclass_path}")

print(f"Total de datasets carregados com sucesso: {len(datasets)}")
print(f"  - Com classes inteiras: {len(datasets_int_classes)}")
print(f"  - Com classes não-inteiras: {len(datasets_non_int_classes)}")
print(f"\nPrimeiros 10 datasets:")
for i, d in enumerate(datasets[:10]):
    classes_type = "(inteiros)" if d['classes_are_int'] else "(não-inteiros)"
    print(f"  {i+1}. {d['name']}: {d['n_samples']} amostras, {d['n_features']} features, {d['n_classes']} classes {classes_type}")

Total de arquivos encontrados: 45

Total de datasets carregados com sucesso: 45
  - Com classes inteiras: 23
  - Com classes não-inteiras: 22

Primeiros 10 datasets:
  1. dataset_10_lymph.csv: 148 amostras, 18 features, 4 classes (não-inteiros)
  2. dataset_11_balance-scale.csv: 625 amostras, 4 features, 3 classes (não-inteiros)
  3. dataset_12_mfeat-factors.csv: 2000 amostras, 216 features, 10 classes (inteiros)
  4. dataset_14_mfeat-fourier.csv: 2000 amostras, 76 features, 10 classes (inteiros)
  5. dataset_16_mfeat-karhunen.csv: 2000 amostras, 64 features, 10 classes (inteiros)
  6. dataset_181_yeast.csv: 1484 amostras, 8 features, 10 classes (não-inteiros)
  7. dataset_182_satimage.csv: 6430 amostras, 36 features, 6 classes (não-inteiros)
  8. dataset_185_baseball.csv: 1340 amostras, 16 features, 3 classes (inteiros)
  9. dataset_186_braziltourism.csv: 412 amostras, 8 features, 7 classes (inteiros)
  10. dataset_187_wine.csv: 178 amostras, 13 features, 3 classes (inteiros)


## 2. Funções Auxiliares

In [79]:
def normalizar_zscore(X_train, X_test):
    """Normaliza usando Z-Score (média 0, desvio padrão 1)"""
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)
    sigma[sigma == 0] = 1e-8
    
    X_train_s = (X_train - mu) / sigma
    X_test_s = (X_test - mu) / sigma
    
    return X_train_s, X_test_s, mu, sigma

# Observação: o carregamento de datasets é realizado diretamente na seção 1,
# para separar os datasets com classes inteiras e os datasets com classes não-inteiras.
# A função de carregamento antiga não é mais utilizada neste notebook.


## 3. Seleção de Datasets para Treino

In [80]:
print(f"Total de datasets encontrados: {len(datasets)}")
print(f"  - Com classes inteiras: {len(datasets_int_classes)}")
print(f"  - Com classes não-inteiras: {len(datasets_non_int_classes)}")

print("\nDatasets com classes inteiras (usar para treinar Logistic e Softmax):")
for d in datasets_int_classes[:10]:
    print(f"  - {d['name']}: {d['n_samples']} amostras, {d['n_features']} features, {d['n_classes']} classes")

print("\nDatasets com classes não-inteiras (usar apenas para treinar Softmax):")
for d in datasets_non_int_classes[:10]:
    print(f"  - {d['name']}: {d['n_samples']} amostras, {d['n_features']} features, {d['n_classes']} classes")

if len(datasets) > 20:
    print(f"\nMostrando apenas os primeiros 10 de cada grupo. Total de datasets: {len(datasets)}")

Total de datasets encontrados: 45
  - Com classes inteiras: 23
  - Com classes não-inteiras: 22

Datasets com classes inteiras (usar para treinar Logistic e Softmax):
  - dataset_12_mfeat-factors.csv: 2000 amostras, 216 features, 10 classes
  - dataset_14_mfeat-fourier.csv: 2000 amostras, 76 features, 10 classes
  - dataset_16_mfeat-karhunen.csv: 2000 amostras, 64 features, 10 classes
  - dataset_185_baseball.csv: 1340 amostras, 16 features, 3 classes
  - dataset_186_braziltourism.csv: 412 amostras, 8 features, 7 classes
  - dataset_187_wine.csv: 178 amostras, 13 features, 3 classes
  - dataset_18_mfeat-morphological.csv: 2000 amostras, 6 features, 10 classes
  - dataset_20_mfeat-pixel.csv: 2000 amostras, 240 features, 10 classes
  - dataset_22_mfeat-zernike.csv: 2000 amostras, 47 features, 10 classes
  - dataset_23_cmc.csv: 1473 amostras, 9 features, 3 classes

Datasets com classes não-inteiras (usar apenas para treinar Softmax):
  - dataset_10_lymph.csv: 148 amostras, 18 features, 4 

## 4. Treino e Comparação dos Algoritmos

In [81]:
def treinar_e_avaliar_modelos(X, y, dataset_name, n_splits=5, lr=0.01, max_iters=1000):
    """
    Treina LogisticRegression e SoftmaxRegression com K-Fold Cross-Validation
    e compara os resultados
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    resultados_logistic = []
    resultados_softmax = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        # 1. Divisão dos dados
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # --- ADICIONAR ESTAS LINHAS AQUI ---
        # Garante que as características são números decimais para permitir cálculos matemáticos
        X_train = np.array(X_train).astype(float)
        X_test = np.array(X_test).astype(float)
        
        # Garante que os rótulos são um array NumPy (já tratámos o mapeamento de strings no y_numeric)
        y_train = np.array(y_train)
        y_test = np.array(y_test)
        # ----------------------------------
        
        # 2. Normalização (agora funcionará porque os dados são float)
        X_train_norm, X_test_norm, _, _ = normalizar_zscore(X_train, X_test)
        
        # ========== LOGISTIC REGRESSION ==========
        try:
            t_inicio = time.time()
            model_lr = LogisticRegression(lr=lr, max_iters=max_iters)
            model_lr.fit(X_train_norm, y_train)
            y_pred_lr = model_lr.predict(X_test_norm)
            tempo_lr = time.time() - t_inicio
            
            acc_lr = accuracy_score(y_test, y_pred_lr)
            prec_lr = precision_score(y_test, y_pred_lr, average='macro', zero_division=0)
            rec_lr = recall_score(y_test, y_pred_lr, average='macro', zero_division=0)
            f1_lr = f1_score(y_test, y_pred_lr, average='macro', zero_division=0)
            
            resultados_logistic.append({
                'fold': fold + 1,
                'accuracy': acc_lr,
                'precision': prec_lr,
                'recall': rec_lr,
                'f1': f1_lr,
                'tempo': tempo_lr
            })
        except Exception as e:
            print(f"Erro no LogisticRegression (Fold {fold+1}): {e}")
            resultados_logistic.append({'fold': fold + 1, 'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0, 'tempo': 0})
        
        # ========== SOFTMAX REGRESSION ==========
        try:
            t_inicio = time.time()
            model_softmax = SoftmaxRegression(lr=lr, max_iters=max_iters)
            model_softmax.fit(X_train_norm, y_train)
            y_pred_softmax = model_softmax.predict(X_test_norm)
            tempo_softmax = time.time() - t_inicio
            
            acc_softmax = accuracy_score(y_test, y_pred_softmax)
            prec_softmax = precision_score(y_test, y_pred_softmax, average='macro', zero_division=0)
            rec_softmax = recall_score(y_test, y_pred_softmax, average='macro', zero_division=0)
            f1_softmax = f1_score(y_test, y_pred_softmax, average='macro', zero_division=0)
            
            resultados_softmax.append({
                'fold': fold + 1,
                'accuracy': acc_softmax,
                'precision': prec_softmax,
                'recall': rec_softmax,
                'f1': f1_softmax,
                'tempo': tempo_softmax
            })
        except Exception as e:
            print(f"Erro no SoftmaxRegression (Fold {fold+1}): {e}")
            resultados_softmax.append({'fold': fold + 1, 'accuracy': 0, 'precision': 0, 'recall': 0, 'f1': 0, 'tempo': 0})
    
    return pd.DataFrame(resultados_logistic), pd.DataFrame(resultados_softmax)

In [82]:
# ================================================================================
# ANÁLISE 1: DATASETS COM CLASSES INTEIRAS - COMPARAR AMBOS ALGORITMOS
# ================================================================================

print("\n" + "="*100)
print("ANÁLISE 1: DATASETS COM CLASSES INTEIRAS - Comparação Logistic vs Softmax")
print("="*100)

resultados_int_classes = []
datasets_int_erro = []

for idx, dataset_info in enumerate(datasets_int_classes):
    print(f"\n[{idx+1}/{len(datasets_int_classes)}] Processando: {dataset_info['name']}")
    print(f"      Amostras: {dataset_info['n_samples']}, Features: {dataset_info['n_features']}, Classes: {dataset_info['n_classes']}")
    
    try:
        df_lr, df_softmax = treinar_e_avaliar_modelos(
            dataset_info['X'], 
            dataset_info['y'], 
            dataset_info['name'],
            n_splits=5,
            lr=0.01,
            max_iters=500
        )
        
        resultados_int_classes.append({
            'dataset': dataset_info['name'],
            'n_classes': dataset_info['n_classes'],
            'n_samples': dataset_info['n_samples'],
            'classes_are_int': True,
            'logistic_accuracy_mean': df_lr['accuracy'].mean(),
            'logistic_accuracy_std': df_lr['accuracy'].std(),
            'logistic_f1_mean': df_lr['f1'].mean(),
            'logistic_f1_std': df_lr['f1'].std(),
            'logistic_precision_mean': df_lr['precision'].mean(),
            'logistic_recall_mean': df_lr['recall'].mean(),
            'logistic_tempo_mean': df_lr['tempo'].mean(),
            'logistic_status': 'sucesso',
            'softmax_accuracy_mean': df_softmax['accuracy'].mean(),
            'softmax_accuracy_std': df_softmax['accuracy'].std(),
            'softmax_f1_mean': df_softmax['f1'].mean(),
            'softmax_f1_std': df_softmax['f1'].std(),
            'softmax_precision_mean': df_softmax['precision'].mean(),
            'softmax_recall_mean': df_softmax['recall'].mean(),
            'softmax_tempo_mean': df_softmax['tempo'].mean(),
            'softmax_status': 'sucesso',
        })
        
        print(f"      ✓ Ambos conseguiram aprender")
        
    except Exception as e:
        print(f"      ✗ Erro: {str(e)}")
        datasets_int_erro.append((dataset_info['name'], str(e)))

df_analise1 = pd.DataFrame(resultados_int_classes)

print("\n" + "-"*100)
print(f"Análise 1 Concluída: {len(resultados_int_classes)}/{len(datasets_int_classes)} datasets processados")
print(f"Datasets com erro: {len(datasets_int_erro)}")
if len(datasets_int_erro) > 0:
    print("Datasets com erro:")
    for name, error in datasets_int_erro:
        print(f"  - {name}: {error}")


ANÁLISE 1: DATASETS COM CLASSES INTEIRAS - Comparação Logistic vs Softmax

[1/23] Processando: dataset_12_mfeat-factors.csv
      Amostras: 2000, Features: 216, Classes: 10
Erro no SoftmaxRegression (Fold 1): tuple index out of range
Erro no SoftmaxRegression (Fold 2): tuple index out of range
Erro no SoftmaxRegression (Fold 3): tuple index out of range
Erro no SoftmaxRegression (Fold 4): tuple index out of range
Erro no SoftmaxRegression (Fold 5): tuple index out of range
      ✓ Ambos conseguiram aprender

[2/23] Processando: dataset_14_mfeat-fourier.csv
      Amostras: 2000, Features: 76, Classes: 10
Erro no SoftmaxRegression (Fold 1): tuple index out of range
Erro no SoftmaxRegression (Fold 2): tuple index out of range
Erro no SoftmaxRegression (Fold 3): tuple index out of range
Erro no SoftmaxRegression (Fold 4): tuple index out of range
Erro no SoftmaxRegression (Fold 5): tuple index out of range
      ✓ Ambos conseguiram aprender

[3/23] Processando: dataset_16_mfeat-karhunen.c

KeyboardInterrupt: 

In [ ]:
# ================================================================================
# ANÁLISE 2: IDENTIFICAR DATASETS ONDE LOGISTIC FALHA MAS SOFTMAX CONSEGUE
# ================================================================================

print("\n" + "="*100)
print("ANÁLISE 2: Datasets onde Logistic NÃO consegue aprender mas Softmax SIM")
print("="*100)

resultados_falhas_logistic = []

for idx, dataset_info in enumerate(datasets_non_int_classes):
    print(f"\n[{idx+1}/{len(datasets_non_int_classes)}] Processando (classes não-inteiras): {dataset_info['name']}")
    print(f"      Amostras: {dataset_info['n_samples']}, Features: {dataset_info['n_features']}, Classes: {dataset_info['n_classes']}")
    print(f"      Classes originais: {dataset_info['classes_original'][:5]}..." if len(dataset_info['classes_original']) > 5 else f"      Classes originais: {dataset_info['classes_original']}")
    
    logistic_status = "falha"
    logistic_accuracy = 0
    logistic_f1 = 0
    logistic_precision = 0
    logistic_recall = 0
    logistic_tempo = 0
    logistic_error = None
    
    softmax_status = "sucesso"
    softmax_accuracy = 0
    softmax_f1 = 0
    softmax_precision = 0
    softmax_recall = 0
    softmax_tempo = 0
    softmax_error = None
    
    # Tentar Logistic Regression (deve falhar com classes não-inteiras)
    try:
        print("      Tentando Logistic Regression...", end=" ")
        df_lr, _ = treinar_e_avaliar_modelos(
            dataset_info['X'], 
            dataset_info['y'], 
            dataset_info['name'],
            n_splits=5,
            lr=0.01,
            max_iters=500
        )
        logistic_status = "sucesso"
        logistic_accuracy = df_lr['accuracy'].mean()
        logistic_f1 = df_lr['f1'].mean()
        logistic_precision = df_lr['precision'].mean()
        logistic_recall = df_lr['recall'].mean()
        logistic_tempo = df_lr['tempo'].mean()
        print("✓ Conseguiu aprender")
    except Exception as e:
        logistic_status = "falha"
        logistic_error = str(e)[:100]
        print(f"✗ Falhou: {logistic_error}")
    
    # Tentar Softmax Regression (deve funcionar mesmo com classes não-inteiras)
    try:
        print("      Tentando Softmax Regression...", end=" ")
        _, df_softmax = treinar_e_avaliar_modelos(
            dataset_info['X'], 
            dataset_info['y'], 
            dataset_info['name'],
            n_splits=5,
            lr=0.01,
            max_iters=500
        )
        softmax_status = "sucesso"
        softmax_accuracy = df_softmax['accuracy'].mean()
        softmax_f1 = df_softmax['f1'].mean()
        softmax_precision = df_softmax['precision'].mean()
        softmax_recall = df_softmax['recall'].mean()
        softmax_tempo = df_softmax['tempo'].mean()
        print("✓ Conseguiu aprender")
    except Exception as e:
        softmax_status = "falha"
        softmax_error = str(e)[:100]
        print(f"✗ Falhou: {softmax_error}")
    
    # Armazenar resultado
    resultados_falhas_logistic.append({
        'dataset': dataset_info['name'],
        'n_classes': dataset_info['n_classes'],
        'n_samples': dataset_info['n_samples'],
        'classes_are_int': False,
        'logistic_status': logistic_status,
        'logistic_accuracy': logistic_accuracy if logistic_status == 'sucesso' else 0,
        'logistic_f1': logistic_f1 if logistic_status == 'sucesso' else 0,
        'logistic_precision': logistic_precision if logistic_status == 'sucesso' else 0,
        'logistic_recall': logistic_recall if logistic_status == 'sucesso' else 0,
        'logistic_tempo': logistic_tempo if logistic_status == 'sucesso' else 0,
        'logistic_error': logistic_error,
        'softmax_status': softmax_status,
        'softmax_accuracy': softmax_accuracy if softmax_status == 'sucesso' else 0,
        'softmax_f1': softmax_f1 if softmax_status == 'sucesso' else 0,
        'softmax_precision': softmax_precision if softmax_status == 'sucesso' else 0,
        'softmax_recall': softmax_recall if softmax_status == 'sucesso' else 0,
        'softmax_tempo': softmax_tempo if softmax_status == 'sucesso' else 0,
        'softmax_error': softmax_error,
    })

df_analise2 = pd.DataFrame(resultados_falhas_logistic)

# Contar estatísticas
logistic_falhas = (df_analise2['logistic_status'] == 'falha').sum()
softmax_sucessos = (df_analise2['softmax_status'] == 'sucesso').sum()

print("\n" + "-"*100)
print(f"Análise 2 Concluída: {len(resultados_falhas_logistic)} datasets processados")
print(f"  - Logistic falhou em: {logistic_falhas} datasets")
print(f"  - Softmax conseguiu aprender em: {softmax_sucessos} datasets")

if logistic_falhas > 0 and logistic_falhas == softmax_sucessos:
    print(f"  ✓ Confirmado: Softmax consegue aprender em TODOS os {logistic_falhas} datasets onde Logistic falha!")

In [ ]:
# ================================================================================
# ANÁLISE 3: RESULTADO GERAL DO SOFTMAX EM TODOS OS DATASETS
# ================================================================================

print("\n" + "="*100)
print("ANÁLISE 3: Desempenho do Softmax em TODOS os datasets de multiclass_classification")
print("="*100)

resultados_softmax_todos = []
datasets_softmax_erro = []

for idx, dataset_info in enumerate(datasets):
    print(f"\n[{idx+1}/{len(datasets)}] Processando: {dataset_info['name']}")
    print(f"      Amostras: {dataset_info['n_samples']}, Features: {dataset_info['n_features']}, Classes: {dataset_info['n_classes']}")
    
    try:
        # Treinando apenas Softmax (usando ambos na função, mas capturando só softmax)
        _, df_softmax = treinar_e_avaliar_modelos(
            dataset_info['X'], 
            dataset_info['y'], 
            dataset_info['name'],
            n_splits=5,
            lr=0.01,
            max_iters=500
        )
        
        resultados_softmax_todos.append({
            'dataset': dataset_info['name'],
            'n_classes': dataset_info['n_classes'],
            'n_samples': dataset_info['n_samples'],
            'n_features': dataset_info['n_features'],
            'classes_are_int': dataset_info['classes_are_int'],
            'softmax_accuracy_mean': df_softmax['accuracy'].mean(),
            'softmax_accuracy_std': df_softmax['accuracy'].std(),
            'softmax_f1_mean': df_softmax['f1'].mean(),
            'softmax_f1_std': df_softmax['f1'].std(),
            'softmax_precision_mean': df_softmax['precision'].mean(),
            'softmax_recall_mean': df_softmax['recall'].mean(),
            'softmax_tempo_mean': df_softmax['tempo'].mean(),
            'status': 'sucesso',
        })
        
        print(f"      ✓ Concluído - Accuracy: {df_softmax['accuracy'].mean():.4f}")
        
    except Exception as e:
        print(f"      ✗ Erro: {str(e)}")
        datasets_softmax_erro.append((dataset_info['name'], str(e)))

df_analise3 = pd.DataFrame(resultados_softmax_todos)

print("\n" + "-"*100)
print(f"Análise 3 Concluída!")
print(f"Datasets processados com sucesso: {len(resultados_softmax_todos)}/{len(datasets)}")
print(f"Datasets com erro: {len(datasets_softmax_erro)}")

print(f"\nResumo Softmax em TODOS os datasets:")
print(f"  - Accuracy média: {df_analise3['softmax_accuracy_mean'].mean():.4f} ± {df_analise3['softmax_accuracy_mean'].std():.4f}")
print(f"  - F1-Score média: {df_analise3['softmax_f1_mean'].mean():.4f}")
print(f"  - Tempo total: {df_analise3['softmax_tempo_mean'].sum():.2f}s")
print(f"  - Datasets com classes inteiras: {(df_analise3['classes_are_int']).sum()}")
print(f"  - Datasets com classes não-inteiras: {(~df_analise3['classes_are_int']).sum()}")

## 5. Resultados da Análise 1 - Datasets com Classes Inteiras

In [ ]:
# ANÁLISE 1: Tabelas Comparativas - Classes Inteiras
print("\n" + "="*100)
print("ANÁLISE 1: COMPARAÇÃO DETALHADA (Datasets com Classes Inteiras)")
print("="*100)

if len(df_analise1) > 0:
    # Calcular deltas
    df_analise1['delta_accuracy'] = df_analise1['softmax_accuracy_mean'] - df_analise1['logistic_accuracy_mean']
    df_analise1['delta_f1'] = df_analise1['softmax_f1_mean'] - df_analise1['logistic_f1_mean']
    df_analise1['winner_accuracy'] = df_analise1['delta_accuracy'].apply(
        lambda x: '✓ Softmax' if x > 0.01 else ('✓ Logistic' if x < -0.01 else '=')
    )
    
    # Tabela de Accuracy
    print("\nComparação de Accuracy:")
    print("-"*100)
    df_acc_table = df_analise1[[
        'dataset', 'n_classes', 'logistic_accuracy_mean', 'softmax_accuracy_mean', 
        'delta_accuracy', 'winner_accuracy'
    ]].copy()
    df_acc_table.columns = ['Dataset', 'Classes', 'Logistic', 'Softmax', 'Δ', 'Vencedor']
    print(df_acc_table.to_string(index=False))
    
    # Estatísticas
    softmax_wins = (df_analise1['delta_accuracy'] > 0.01).sum()
    logistic_wins = (df_analise1['delta_accuracy'] < -0.01).sum()
    empates = (abs(df_analise1['delta_accuracy']) <= 0.01).sum()
    
    print(f"\nEstatísticas da Análise 1:")
    print(f"  - Softmax venceu em Accuracy: {softmax_wins}/{len(df_analise1)} datasets ({softmax_wins/len(df_analise1)*100:.1f}%)")
    print(f"  - Logistic venceu em Accuracy: {logistic_wins}/{len(df_analise1)} datasets ({logistic_wins/len(df_analise1)*100:.1f}%)")
    print(f"  - Empates: {empates}/{len(df_analise1)} datasets ({empates/len(df_analise1)*100:.1f}%)")
    
    print(f"\nMédias Gerais (Análise 1):")
    print(f"  Logistic Regression:")
    print(f"    - Accuracy: {df_analise1['logistic_accuracy_mean'].mean():.4f} ± {df_analise1['logistic_accuracy_mean'].std():.4f}")
    print(f"    - F1-Score: {df_analise1['logistic_f1_mean'].mean():.4f} ± {df_analise1['logistic_f1_mean'].std():.4f}")
    print(f"    - Tempo médio: {df_analise1['logistic_tempo_mean'].mean():.4f}s")
    
    print(f"\n  Softmax Regression:")
    print(f"    - Accuracy: {df_analise1['softmax_accuracy_mean'].mean():.4f} ± {df_analise1['softmax_accuracy_mean'].std():.4f}")
    print(f"    - F1-Score: {df_analise1['softmax_f1_mean'].mean():.4f} ± {df_analise1['softmax_f1_mean'].std():.4f}")
    print(f"    - Tempo médio: {df_analise1['softmax_tempo_mean'].mean():.4f}s")
    
    print(f"\n  Ganho Médio do Softmax:")
    print(f"    - Accuracy: {df_analise1['delta_accuracy'].mean():.4f}")
    print(f"    - F1-Score: {df_analise1['delta_f1'].mean():.4f}")
else:
    print("Nenhum dataset com classes inteiras encontrado")

## 5.5. Visualizações da Análise 1

In [ ]:
# Visualizações Análise 1 - Datasets com Classes Inteiras
if len(df_analise1) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Gráfico 1: Distribuição de Accuracy
    ax1 = axes[0, 0]
    ax1.hist(df_analise1['logistic_accuracy_mean'], bins=12, alpha=0.6, label='Logistic', color='#3498db', edgecolor='black')
    ax1.hist(df_analise1['softmax_accuracy_mean'], bins=12, alpha=0.6, label='Softmax', color='#e74c3c', edgecolor='black')
    ax1.set_xlabel('Accuracy')
    ax1.set_ylabel('Frequência')
    ax1.set_title('Distribuição de Accuracy (Classes Inteiras)', fontweight='bold', fontsize=12)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Gráfico 2: Distribuição de F1-Score
    ax2 = axes[0, 1]
    ax2.hist(df_analise1['logistic_f1_mean'], bins=12, alpha=0.6, label='Logistic', color='#3498db', edgecolor='black')
    ax2.hist(df_analise1['softmax_f1_mean'], bins=12, alpha=0.6, label='Softmax', color='#e74c3c', edgecolor='black')
    ax2.set_xlabel('F1-Score')
    ax2.set_ylabel('Frequência')
    ax2.set_title('Distribuição de F1-Score (Classes Inteiras)', fontweight='bold', fontsize=12)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Gráfico 3: Box Plot Accuracy
    ax3 = axes[1, 0]
    bp = ax3.boxplot([df_analise1['logistic_accuracy_mean'], df_analise1['softmax_accuracy_mean']], 
                      labels=['Logistic', 'Softmax'], patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax3.set_ylabel('Accuracy')
    ax3.set_title('Distribuição de Accuracy (Box Plot)', fontweight='bold', fontsize=12)
    ax3.grid(axis='y', alpha=0.3)
    
    # Gráfico 4: Scatter - Logistic vs Softmax
    ax4 = axes[1, 1]
    ax4.scatter(df_analise1['logistic_accuracy_mean'], df_analise1['softmax_accuracy_mean'],
               s=120, alpha=0.6, c=df_analise1['n_classes'], cmap='viridis', edgecolors='black', linewidth=1.5)
    ax4.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=2, label='Perfeita Igualdade')
    ax4.set_xlabel('Logistic Regression Accuracy')
    ax4.set_ylabel('Softmax Regression Accuracy')
    ax4.set_title('Logistic vs Softmax - Accuracy', fontweight='bold', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim(-0.05, 1.05)
    ax4.set_ylim(-0.05, 1.05)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizações da Análise 1 geradas com sucesso")

In [ ]:
## 6. Resultados da Análise 2 - Datasets onde Logistic Falha mas Softmax Consegue

In [ ]:
# ANÁLISE 2: Detalhamento - Onde Logistic Falha
print("\n" + "="*100)
print("ANÁLISE 2: DATASETS ONDE LOGISTIC FALHA MAS SOFTMAX CONSEGUE")
print("="*100)

if len(df_analise2) > 0:
    # Contar
    logistic_falhas = (df_analise2['logistic_status'] == 'falha').sum()
    softmax_sucessos = (df_analise2['softmax_status'] == 'sucesso').sum()
    ambos_sucessos = ((df_analise2['logistic_status'] == 'sucesso') & (df_analise2['softmax_status'] == 'sucesso')).sum()
    
    print(f"\nResumo Análise 2 (Datasets com Classes NÃO-inteiras):")
    print(f"  Total de datasets: {len(df_analise2)}")
    print(f"  - Logistic conseguiu aprender: {len(df_analise2) - logistic_falhas} ({(len(df_analise2) - logistic_falhas)/len(df_analise2)*100:.1f}%)")
    print(f"  - Logistic FALHOU: {logistic_falhas} ({logistic_falhas/len(df_analise2)*100:.1f}%)")
    print(f"  - Softmax conseguiu aprender: {softmax_sucessos} ({softmax_sucessos/len(df_analise2)*100:.1f}%)")
    print(f"\n✓ VANTAGEM DO SOFTMAX:")
    print(f"  Em {logistic_falhas} datasets, apenas Softmax conseguiu aprender!")
    print(f"  (Logistic não consegue lidar com classes não-inteiras)")
    
    # Tabela de Datasets onde Logistic Falha e Softmax Consegue
    df_analise2_falhas = df_analise2[
        (df_analise2['logistic_status'] == 'falha') & 
        (df_analise2['softmax_status'] == 'sucesso')
    ].copy()
    
    if len(df_analise2_falhas) > 0:
        print(f"\nDatasets onde Logistic FALHA mas Softmax SIM consegue:")
        print("-"*100)
        df_table = df_analise2_falhas[[
            'dataset', 'n_classes', 'logistic_status', 'softmax_accuracy', 'softmax_f1', 'softmax_status'
        ]].copy()
        df_table.columns = ['Dataset', 'Classes', 'Logistic', 'Softmax Acc', 'Softmax F1', 'Softmax Status']
        print(df_table.to_string(index=False))
        
        print(f"\n  Accuracy média do Softmax nesses datasets: {df_analise2_falhas['softmax_accuracy'].mean():.4f}")
        print(f"  F1-Score médio do Softmax nesses datasets: {df_analise2_falhas['softmax_f1'].mean():.4f}")
    
    # Tabela de Datasets onde Ambos conseguem aprender
    df_analise2_ambos = df_analise2[
        (df_analise2['logistic_status'] == 'sucesso') & 
        (df_analise2['softmax_status'] == 'sucesso')
    ].copy()
    
    if len(df_analise2_ambos) > 0:
        print(f"\n\nDatasets com classes NÃO-inteiras onde AMBOS conseguem aprender ({len(df_analise2_ambos)}):")
        print("-"*100)
        df_analise2_ambos['delta_accuracy'] = df_analise2_ambos['softmax_accuracy'] - df_analise2_ambos['logistic_accuracy']
        df_table = df_analise2_ambos[[
            'dataset', 'n_classes', 'logistic_accuracy', 'softmax_accuracy', 'delta_accuracy'
        ]].copy()
        df_table.columns = ['Dataset', 'Classes', 'Logistic', 'Softmax', 'Δ']
        print(df_table.to_string(index=False))
        
        softmax_wins = (df_analise2_ambos['delta_accuracy'] > 0.01).sum()
        print(f"\n  Softmax venceu em {softmax_wins}/{len(df_analise2_ambos)} datasets mesmo com classes não-inteiras")
else:
    print("Nenhum dataset com classes não-inteiras encontrado")

In [ ]:
# Visualizações Análise 2
if len(df_analise2) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Gráfico 1: Status dos Algoritmos
    ax1 = axes[0]
    logistic_falhas = (df_analise2['logistic_status'] == 'falha').sum()
    softmax_sucessos = (df_analise2['softmax_status'] == 'sucesso').sum()
    logistic_sucessos = len(df_analise2) - logistic_falhas
    softmax_falhas = len(df_analise2) - softmax_sucessos
    
    x_pos = np.arange(2)
    width = 0.35
    
    bars1 = ax1.bar(x_pos - width/2, [logistic_sucessos, logistic_falhas], width, 
                    label='Logistic', color=['#2ecc71', '#e74c3c'], alpha=0.8, edgecolor='black')
    bars2 = ax1.bar(x_pos + width/2, [softmax_sucessos, softmax_falhas], width,
                    label='Softmax', color=['#3498db', '#95a5a6'], alpha=0.8, edgecolor='black')
    
    ax1.set_ylabel('Quantidade de Datasets')
    ax1.set_title('Status dos Algoritmos (Classes Não-inteiras)', fontweight='bold', fontsize=12)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(['Sucesso', 'Falha'])
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Adicionar valores nas barras
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 2: Pie Chart - Sucesso Softmax
    ax2 = axes[1]
    sucesso_labels = ['Softmax Sucesso', 'Softmax Falha']
    sucesso_sizes = [softmax_sucessos, softmax_falhas]
    colors = ['#2ecc71', '#95a5a6']
    
    wedges, texts, autotexts = ax2.pie(sucesso_sizes, labels=sucesso_labels, autopct='%1.1f%%',
                                        colors=colors, startangle=90, textprops={'fontsize': 11})
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    ax2.set_title('Taxa de Sucesso do Softmax\n(Classes Não-inteiras)', fontweight='bold', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizações da Análise 2 geradas com sucesso")

## 7. Resultados da Análise 3 - Softmax em Todos os Datasets

In [ ]:
# ANÁLISE 3: Detalhamento Completo - Softmax em Todos os Datasets
print("\n" + "="*100)
print("ANÁLISE 3: DESEMPENHO GERAL DO SOFTMAX EM TODOS OS DATASETS")
print("="*100)

if len(df_analise3) > 0:
    # Separar por tipo de classe
    df_a3_int = df_analise3[df_analise3['classes_are_int'] == True]
    df_a3_non_int = df_analise3[df_analise3['classes_are_int'] == False]
    
    print(f"\nTotal de datasets processados: {len(df_analise3)}")
    print(f"  - Com classes inteiras: {len(df_a3_int)}")
    print(f"  - Com classes não-inteiras: {len(df_a3_non_int)}")
    
    print(f"\nDesempenho GERAL do Softmax (Todos os datasets):")
    print(f"  - Accuracy: {df_analise3['softmax_accuracy_mean'].mean():.4f} ± {df_analise3['softmax_accuracy_mean'].std():.4f}")
    print(f"  - F1-Score: {df_analise3['softmax_f1_mean'].mean():.4f} ± {df_analise3['softmax_f1_mean'].std():.4f}")
    print(f"  - Precision: {df_analise3['softmax_precision_mean'].mean():.4f}")
    print(f"  - Recall: {df_analise3['softmax_recall_mean'].mean():.4f}")
    print(f"  - Tempo total: {df_analise3['softmax_tempo_mean'].sum():.2f}s")
    print(f"  - Tempo médio por dataset: {df_analise3['softmax_tempo_mean'].mean():.4f}s")
    
    print(f"\nDesempenho por tipo de classe:")
    print(f"  Classes Inteiras (n={len(df_a3_int)}):")
    if len(df_a3_int) > 0:
        print(f"    - Accuracy: {df_a3_int['softmax_accuracy_mean'].mean():.4f}")
        print(f"    - F1-Score: {df_a3_int['softmax_f1_mean'].mean():.4f}")
    
    print(f"  Classes Não-inteiras (n={len(df_a3_non_int)}):")
    if len(df_a3_non_int) > 0:
        print(f"    - Accuracy: {df_a3_non_int['softmax_accuracy_mean'].mean():.4f}")
        print(f"    - F1-Score: {df_a3_non_int['softmax_f1_mean'].mean():.4f}")
    
    # Top 10 Melhores e Piores
    print(f"\nTop 10 Melhores Desempenhos (Softmax):")
    print("-"*100)
    df_top10 = df_analise3.nlargest(10, 'softmax_accuracy_mean')[
        ['dataset', 'n_classes', 'n_samples', 'softmax_accuracy_mean', 'softmax_f1_mean']
    ].copy()
    df_top10.columns = ['Dataset', 'Classes', 'Amostras', 'Accuracy', 'F1-Score']
    print(df_top10.to_string(index=False))
    
    print(f"\nTop 10 Piores Desempenhos (Softmax):")
    print("-"*100)
    df_bottom10 = df_analise3.nsmallest(10, 'softmax_accuracy_mean')[
        ['dataset', 'n_classes', 'n_samples', 'softmax_accuracy_mean', 'softmax_f1_mean']
    ].copy()
    df_bottom10.columns = ['Dataset', 'Classes', 'Amostras', 'Accuracy', 'F1-Score']
    print(df_bottom10.to_string(index=False))
    
    print(f"\nAnálise por Número de Classes:")
    print("-"*100)
    for n_classes in sorted(df_analise3['n_classes'].unique()):
        df_subset = df_analise3[df_analise3['n_classes'] == n_classes]
        print(f"  {n_classes} classes: {len(df_subset)} datasets | Accuracy: {df_subset['softmax_accuracy_mean'].mean():.4f} | F1: {df_subset['softmax_f1_mean'].mean():.4f}")
else:
    print("Nenhum resultado na Análise 3")

In [ ]:
# Visualizações Análise 3 - Softmax em Todos os Datasets
if len(df_analise3) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Gráfico 1: Distribuição de Accuracy (Todos os datasets)
    ax1 = axes[0, 0]
    ax1.hist(df_analise3['softmax_accuracy_mean'], bins=20, alpha=0.7, color='#3498db', edgecolor='black')
    ax1.axvline(df_analise3['softmax_accuracy_mean'].mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {df_analise3["softmax_accuracy_mean"].mean():.4f}')
    ax1.set_xlabel('Accuracy')
    ax1.set_ylabel('Frequência')
    ax1.set_title('Distribuição de Accuracy - Softmax (Todos os Datasets)', fontweight='bold', fontsize=12)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Gráfico 2: Distribuição de F1-Score
    ax2 = axes[0, 1]
    ax2.hist(df_analise3['softmax_f1_mean'], bins=20, alpha=0.7, color='#2ecc71', edgecolor='black')
    ax2.axvline(df_analise3['softmax_f1_mean'].mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {df_analise3["softmax_f1_mean"].mean():.4f}')
    ax2.set_xlabel('F1-Score')
    ax2.set_ylabel('Frequência')
    ax2.set_title('Distribuição de F1-Score - Softmax (Todos os Datasets)', fontweight='bold', fontsize=12)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Gráfico 3: Accuracy vs Número de Classes
    ax3 = axes[1, 0]
    class_counts = df_analise3.groupby('n_classes')['softmax_accuracy_mean'].apply(list)
    positions = list(class_counts.index)
    ax3.boxplot([class_counts[pos] for pos in positions], positions=positions, widths=0.6, patch_artist=True)
    ax3.set_xlabel('Número de Classes')
    ax3.set_ylabel('Accuracy')
    ax3.set_title('Accuracy por Número de Classes', fontweight='bold', fontsize=12)
    ax3.grid(axis='y', alpha=0.3)
    
    # Gráfico 4: Accuracy vs Número de Amostras
    ax4 = axes[1, 1]
    scatter = ax4.scatter(df_analise3['n_samples'], df_analise3['softmax_accuracy_mean'],
                         s=100, alpha=0.6, c=df_analise3['n_classes'], cmap='viridis',
                         edgecolors='black', linewidth=1)
    ax4.set_xlabel('Número de Amostras')
    ax4.set_ylabel('Accuracy')
    ax4.set_title('Accuracy vs Tamanho do Dataset', fontweight='bold', fontsize=12)
    ax4.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label('Número de Classes')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizações da Análise 3 geradas com sucesso")

## 8. Resumo Executivo e Conclusões

In [83]:
# Tabela consolidada - Resumo por Dataset
df_tabela = df_comparacao[[
    'dataset', 'n_classes', 'n_samples', 'n_features',
    'logistic_accuracy_mean', 'softmax_accuracy_mean',
    'logistic_f1_mean', 'softmax_f1_mean',
    'logistic_tempo_mean', 'softmax_tempo_mean'
]].copy()

df_tabela.columns = [
    'Dataset', 'Classes', 'Amostras', 'Features',
    'Log_Acc', 'Soft_Acc', 'Log_F1', 'Soft_F1', 'Log_Tempo', 'Soft_Tempo'
]

df_tabela['Δ_Acc'] = df_tabela['Soft_Acc'] - df_tabela['Log_Acc']
df_tabela['Δ_F1'] = df_tabela['Soft_F1'] - df_tabela['Log_F1']
df_tabela['Winner_Acc'] = df_tabela['Δ_Acc'].apply(lambda x: '✓ Soft' if x > 0.01 else ('✓ Log' if x < -0.01 else '='))

# Ordenar por diferença de accuracy
df_tabela_sorted = df_tabela.sort_values('Δ_Acc', ascending=False)

print("\n" + "="*120)
print("TABELA RESUMIDA: COMPARAÇÃO POR DATASET (Ordenada por Ganho Softmax em Accuracy)")
print("="*120)

# Mostrar top 20 melhores e piores
print("\nTop 20 Datasets onde Softmax ganhou:")
display(df_tabela_sorted[['Dataset', 'Classes', 'Log_Acc', 'Soft_Acc', 'Δ_Acc', 'Winner_Acc']].head(20))

print("\n\nTop 20 Datasets onde Logistic ganhou:")
display(df_tabela_sorted[['Dataset', 'Classes', 'Log_Acc', 'Soft_Acc', 'Δ_Acc', 'Winner_Acc']].tail(20))

NameError: name 'df_comparacao' is not defined

In [ ]:
print("\n" + "="*80)
print("CONCLUSÕES E RECOMENDAÇÕES")
print("="*80)

conclusoes = f"""
RESUMO EXECUTIVO (Baseado em {len(df_comparacao)} Datasets):

1. **Superioridade do Softmax para Multiclasse:**
   - Softmax venceu em {softmax_wins_acc} datasets ({softmax_wins_acc/len(df_comparacao)*100:.1f}%)
   - Logistic venceu em {logistic_wins_acc} datasets ({logistic_wins_acc/len(df_comparacao)*100:.1f}%)
   - Ganho médio do Softmax: {acc_delta.mean():.4f} em Accuracy
   - Confirmando que Softmax é teoricamente superior para problemas multiclasse

2. **Desempenho em Accuracy:**
   - Logistic: {df_comparacao['logistic_accuracy_mean'].mean():.4f} ± {df_comparacao['logistic_accuracy_mean'].std():.4f}
   - Softmax:  {df_comparacao['softmax_accuracy_mean'].mean():.4f} ± {df_comparacao['softmax_accuracy_mean'].std():.4f}
   - Melhoria: {(df_comparacao['softmax_accuracy_mean'].mean() - df_comparacao['logistic_accuracy_mean'].mean())*100:.2f}%

3. **Desempenho em F1-Score:**
   - Logistic: {df_comparacao['logistic_f1_mean'].mean():.4f} ± {df_comparacao['logistic_f1_mean'].std():.4f}
   - Softmax:  {df_comparacao['softmax_f1_mean'].mean():.4f} ± {df_comparacao['softmax_f1_mean'].std():.4f}
   - Softmax venceu em {softmax_wins_f1} datasets ({softmax_wins_f1/len(df_comparacao)*100:.1f}%)

4. **Custo Computacional:**
   - Logistic: {df_comparacao['logistic_tempo_mean'].mean():.4f}s (média)
   - Softmax:  {df_comparacao['softmax_tempo_mean'].mean():.4f}s (média)
   - Diferença: {abs(tempo_delta.mean()):.4f}s por dataset
   - Mais rápido: Logistic (trade-off com precisão)

5. **Estabilidade de Desempenho:**
   - Logistic std: {df_comparacao['logistic_accuracy_mean'].std():.4f}
   - Softmax std:  {df_comparacao['softmax_accuracy_mean'].std():.4f}
   - Softmax mais estável/consistente: {'Sim' if df_comparacao['softmax_accuracy_mean'].std() < df_comparacao['logistic_accuracy_mean'].std() else 'Não'}

RECOMENDAÇÕES FINAIS:
✓ Use SOFTMAX para classificação multiclasse (estatisticamente superior)
✓ Use LOGISTIC apenas se velocidade for crítica e qualidade aceitável
✓ Softmax oferece melhor equilíbrio entre Precision e Recall
✓ Considere ensemble dos dois modelos para máxima robustez
✓ Ajuste hiperparâmetros (learning_rate, max_iterations) conforme necessário
"""

print(conclusoes)